In [ ]:
import cv2
import numpy as np
import torch
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.ops import nms

# Load the pre-trained RetinaNet model
model = retinanet_resnet50_fpn(pretrained=True)
model.eval()

cap = cv2.VideoCapture('C:/Users/HP/Py Code/Neural Network/Pytorch/Object detection/v1.mp4')

# Define the Tracker class to track the detected person objects based on the proximity between center points of bounding boxes
class Tracker:
    def __init__(self):
        # Store the center positions of the objects
        self.center_points = {}
        # Keep the count of the IDs
        # each time a new object id detected, the count will increase by one
        self.id_count = 0

    def update(self, objects_rect):
        # Objects boxes and ids
        objects_bbs_ids = []

        # Get center point of new object
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + x + w) // 2
            cy = (y + y + h) // 2

            # Find out if that object was detected already
            same_object_detected = False
            for id, pt in self.center_points.items():
                dist = np.hypot(cx - pt[0], cy - pt[1])

                if dist < 35:
                    self.center_points[id] = (cx, cy)
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break

            # New object is detected we assign the ID to that object
            if same_object_detected is False:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1

        # Clean the dictionary by center points to remove IDS not used anymore
        new_center_points = {}
        for obj_bb_id in objects_bbs_ids:
            _, _, _, _, object_id = obj_bb_id
            center = self.center_points[object_id]
            new_center_points[object_id] = center

        # Update dictionary with IDs not used removed
        self.center_points = new_center_points.copy()
        return objects_bbs_ids

# Define the region of interest polygon
area_1 = [(15, 8), (1000, 8), (1000, 500), (15, 500)]

# Initialize the tracker
tracker = Tracker()

# Define a set to store person IDs within the region of interest
area1 = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (1020, 500))

    # Draw the region of interest polygon
    cv2.polylines(frame, [np.array(area_1, np.int32)], True, (0, 255, 0), 3)

    # Convert frame to torch tensor
    tensor_frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

    # Perform inference
    with torch.no_grad():
        predictions = model([tensor_frame])

    # Extract bounding boxes of detected 'person' objects
    detected_boxes = predictions[0]['boxes'].cpu().numpy()
    detected_scores = predictions[0]['scores'].cpu().numpy()
    detected_classes = predictions[0]['labels'].cpu().numpy()
    
    # Filter out 'person' class detections with scores above a certain threshold
    person_boxes = detected_boxes[(detected_classes == 1) & (detected_scores > 0.7)]  # 1 corresponds to 'person' class

    # Apply non-maximum suppression (NMS) to remove redundant bounding boxes
    if len(person_boxes) > 0:
        keep = nms(torch.tensor(person_boxes.astype(np.float32)), torch.tensor(detected_scores[(detected_classes == 1) & (detected_scores > 0.7)]), iou_threshold=0.5)
        person_boxes = person_boxes[keep]

    objects_rect = []
    for box in person_boxes:
        x1, y1, x2, y2 = box.astype(int)
        objects_rect.append([x1, y1, x2 - x1, y2 - y1])

    # Update tracker with detected bounding boxes
    boxes_ids = tracker.update(objects_rect)

    area1.clear()
    for box_id in boxes_ids:
        x, y, w, h, id = box_id
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
        cv2.putText(frame, str(id), (x, y), cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 0), 2)

        # Check if the object is within the defined region of interest
        result = cv2.pointPolygonTest(np.array(area_1, np.int32), (int((x + w) / 2), int((y + h) / 2)), False)
        if result > 0:
            area1.add(id)

    p = len(area1)
    print(p)
    cv2.putText(frame, 'count:' + str(p), (20, 30), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)
    if p > 5:
        cv2.putText(frame, 'Overloaded', (20, 60), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)

    cv2.imshow('FRAME', frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
